# PlantAdvisor — исходный baseline

Архив самостоятельного Colab-эксперимента. Для запуска используйте [рабочий ноутбук](../baseline_colab.ipynb).


## 1. Зависимости и окружение


In [ ]:
%pip -q install "transformers>=4.45,<5" "accelerate>=1.0,<2" "bitsandbytes>=0.45,<1" "pandas>=2,<3" "matplotlib>=3.7,<4" "safetensors>=0.4"

In [ ]:
import os, sys, json, random, re, time, hashlib, subprocess
import importlib.metadata
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
assert torch.cuda.is_available(), "Выберите GPU в Colab: Среда выполнения → Сменить среду выполнения → T4"
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

PROJECT = Path('/content/plant_advisor_rk1')
DATA = PROJECT / 'data'
OUTPUT = PROJECT / 'outputs'
DATA.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

LIB_VERSIONS = {p: importlib.metadata.version(p) for p in
    ['transformers', 'accelerate', 'bitsandbytes', 'huggingface-hub', 'safetensors']}
LIB_VERSIONS['torch'] = torch.__version__
print('Версии:', LIB_VERSIONS)

# Полный список установленных пакетов для повторения окружения.
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'],
                        capture_output=True, text=True, check=True)
(OUTPUT / 'pip_freeze.txt').write_text(freeze.stdout, encoding='utf-8')


## 2. Тестовая выборка

30 вопросов: уход и диагностика. Не использовать для обучения.


In [ ]:
EVAL_QUESTIONS = [
  {
    "id": "q01",
    "task": "care",
    "question": "Как ухаживать за монстерой в квартире с восточным окном?"
  },
  {
    "id": "q02",
    "task": "diagnosis",
    "question": "У монстеры желтеют листья, а грунт постоянно влажный. Что проверить?"
  },
  {
    "id": "q03",
    "task": "care",
    "question": "Как понять, что замиокулькас пора поливать зимой?"
  },
  {
    "id": "q04",
    "task": "diagnosis",
    "question": "У фикуса Бенджамина резко опадают листья после перестановки. Что могло случиться?"
  },
  {
    "id": "q05",
    "task": "care",
    "question": "Какие условия нужны спатифиллуму, чтобы он нормально рос?"
  },
  {
    "id": "q06",
    "task": "diagnosis",
    "question": "У спатифиллума коричневые кончики листьев. В чём могут быть причины?"
  },
  {
    "id": "q07",
    "task": "care",
    "question": "Можно ли держать сансевиерию в глубине комнаты без прямого света?"
  },
  {
    "id": "q08",
    "task": "diagnosis",
    "question": "У сансевиерии основание листа стало мягким и водянистым. Что предпринять?"
  },
  {
    "id": "q09",
    "task": "care",
    "question": "Как пересадить молодой хлорофитум, чтобы не повредить корни?"
  },
  {
    "id": "q10",
    "task": "diagnosis",
    "question": "У хлорофитума побледнели листья и замедлился рост. Что проверить?"
  },
  {
    "id": "q11",
    "task": "care",
    "question": "Как ухаживать за орхидеей фаленопсис после цветения?"
  },
  {
    "id": "q12",
    "task": "diagnosis",
    "question": "У фаленопсиса сморщились листья, а корни в прозрачном горшке серебристые. Как разобраться?"
  },
  {
    "id": "q13",
    "task": "care",
    "question": "Как выбрать место для кактуса зимой в квартире?"
  },
  {
    "id": "q14",
    "task": "diagnosis",
    "question": "У кактуса появилась мягкая тёмная зона около основания. Какие действия безопасны?"
  },
  {
    "id": "q15",
    "task": "care",
    "question": "Как правильно поливать алоэ в разные сезоны?"
  },
  {
    "id": "q16",
    "task": "diagnosis",
    "question": "Листья алоэ стали тонкими и начали скручиваться. Какие причины возможны?"
  },
  {
    "id": "q17",
    "task": "care",
    "question": "Что учитывать при уходе за калатеей в сухой квартире?"
  },
  {
    "id": "q18",
    "task": "diagnosis",
    "question": "У калатеи скручиваются листья и сохнут края. Что сначала проверить?"
  },
  {
    "id": "q19",
    "task": "care",
    "question": "Когда и чем подкармливать пеларгонию на подоконнике?"
  },
  {
    "id": "q20",
    "task": "diagnosis",
    "question": "У пеларгонии желтеют нижние листья и отваливаются бутоны. Как действовать?"
  },
  {
    "id": "q21",
    "task": "care",
    "question": "Как подобрать субстрат для антуриума при пересадке?"
  },
  {
    "id": "q22",
    "task": "diagnosis",
    "question": "У антуриума на листьях появились тёмные пятна, но причина неизвестна. Что нужно уточнить?"
  },
  {
    "id": "q23",
    "task": "care",
    "question": "Как ухаживать за драценой зимой рядом с батареей?"
  },
  {
    "id": "q24",
    "task": "diagnosis",
    "question": "У драцены сохнут кончики листьев. Полив регулярный, других данных нет. Что спросить?"
  },
  {
    "id": "q25",
    "task": "care",
    "question": "Что нужно знать о выращивании толстянки на северном окне?"
  },
  {
    "id": "q26",
    "task": "diagnosis",
    "question": "У толстянки листья стали мягкими, хотя почва сырая. В чём риск?"
  },
  {
    "id": "q27",
    "task": "care",
    "question": "Как обеспечить уход за плющом комнатным в летнюю жару?"
  },
  {
    "id": "q28",
    "task": "diagnosis",
    "question": "На листьях плюща появилась мелкая светлая крапчатость и тонкая паутинка. Что проверить?"
  },
  {
    "id": "q29",
    "task": "care",
    "question": "Как выбрать горшок и дренаж для диффенбахии?"
  },
  {
    "id": "q30",
    "task": "diagnosis",
    "question": "У диффенбахии внезапно поникли листья, хотя вчера выглядела нормально. Каков порядок проверки?"
  }
]
assert len(EVAL_QUESTIONS) == 30
assert len({x['id'] for x in EVAL_QUESTIONS}) == 30
assert {x['task'] for x in EVAL_QUESTIONS} == {'care', 'diagnosis'}
assert len({x['id'] for x in EVAL_QUESTIONS}) == 30
assert {x['task'] for x in EVAL_QUESTIONS} == {'care', 'diagnosis'}
with (DATA/'eval_questions.jsonl').open('w',encoding='utf-8') as f:
    for rec in EVAL_QUESTIONS:
        f.write(json.dumps(rec,ensure_ascii=False)+'\n')
pd.DataFrame(EVAL_QUESTIONS).groupby('task').size()

## 3. Конфигурация и загрузка модели


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
SYSTEM_PROMPT = """Ты русскоязычный консультант по домашним растениям.
Давай конкретные, практически полезные и осторожные рекомендации.

Если пользователь спрашивает об уходе, выдай ответ строго с разделами
в указанном порядке:
Условия
Рекомендации

Если пользователь описывает симптомы или проблему с растением,
выдай ответ строго с разделами в указанном порядке:
Возможные причины
Что проверить
Что делать

Пиши каждый заголовок на отдельной строке; после заголовка обязательно
добавляй содержательный текст или пункты. Не используй заголовок как часть
предложения. Не выдавай предположение за установленный диагноз.
Когда информации недостаточно, предложи, какие данные нужно уточнить.
Избегай опасных, безосновательных и чрезмерно категоричных советов."""
MAX_NEW_TOKENS = 512
FORMAT_CHECK_VERSION = 'v2_standalone_ordered_nonempty_sections'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
)
model.eval()
print('Loaded model:', MODEL_ID)
print('Device map:', model.hf_device_map)

# Отслеживаем реальную ревизию весов, а не только строку MODEL_ID.
model_commit = getattr(model.config, '_commit_hash', None)
if not model_commit:
    from huggingface_hub import model_info
    model_commit = model_info(MODEL_ID).sha
assert model_commit, 'Не удалось определить ревизию модели'

q_serialized = json.dumps(EVAL_QUESTIONS, ensure_ascii=False,
                          sort_keys=True, separators=(',', ':'))
questions_sha256 = hashlib.sha256(q_serialized.encode('utf-8')).hexdigest()
RUN_CONFIG = {
    'experiment': 'plant_advisor_baseline',
    'model_id': MODEL_ID,
    'model_revision': model_commit,
    'system_prompt': SYSTEM_PROMPT,
    'max_new_tokens': MAX_NEW_TOKENS,
    'decoding': {'do_sample': False, 'num_beams': 1},
    'quantization': {'bits': 4, 'type': 'nf4', 'double_quant': True,
                     'compute_dtype': 'float16'},
    'seed': SEED,
    'question_sha256': questions_sha256,
    'format_check_version': FORMAT_CHECK_VERSION,
    'library_versions': LIB_VERSIONS,
}
canonical_config = json.dumps(RUN_CONFIG, ensure_ascii=False, sort_keys=True,
                              separators=(',', ':'))
RUN_ID = hashlib.sha256(canonical_config.encode('utf-8')).hexdigest()[:16]
RUN_DIR = OUTPUT / 'runs' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RUN_DIR / 'config.json'
if CONFIG_PATH.exists():
    existing_config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    if existing_config != RUN_CONFIG:
        raise RuntimeError('Конфликт конфигурации: уже существует папка с другим запуском')
else:
    CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False),
                           encoding='utf-8')

baseline_path = RUN_DIR / 'baseline_answers.jsonl'
print('RUN_ID:', RUN_ID)
print('Данные этого запуска:', RUN_DIR)


## 4. Проверка на двух вопросах


In [ ]:
@torch.inference_mode()
def answer_question(question):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_tensors='pt'
    ).to('cuda')
    output = model.generate(
        inputs, max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False, num_beams=1,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output[0, inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

quick_run = []
for item in [EVAL_QUESTIONS[0], EVAL_QUESTIONS[1]]:
    start = time.perf_counter()
    response = answer_question(item['question'])
    rec = {**item, 'answer': response, 'seconds': round(time.perf_counter()-start, 2),
           'run_id': RUN_ID}
    quick_run.append(rec)
    print('\n' + '='*60, '\n', item['id'], item['question'], '\n\n', response)
with (RUN_DIR / 'quick_run.jsonl').open('w', encoding='utf-8') as f:
    for rec in quick_run:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')


## 5. Форматная метрика

Проверяется наличие, порядок и непустота обязательных разделов.


In [ ]:
EXPECTED_SECTIONS = {
    'care': ['условия', 'рекомендации'],
    'diagnosis': ['возможные причины', 'что проверить', 'что делать'],
}


def normalize_heading(line):
    """Заголовок на отдельной строке: допускаем Markdown-разметку и нумерацию."""
    cleaned = line.strip().casefold().replace('ё', 'е')
    cleaned = re.sub(r'^#{1,6}\s*', '', cleaned)
    cleaned = re.sub(r'^\d{1,2}[.)]\s*', '', cleaned)
    cleaned = re.sub(r'[*_`]', '', cleaned).strip()
    return cleaned.removesuffix(':').strip()


def format_check(text, task):
    if task not in EXPECTED_SECTIONS:
        raise ValueError(f'Неизвестный тип запроса: {task}')
    expected = EXPECTED_SECTIONS[task]
    lines = text.splitlines()
    detected = [(i, normalize_heading(s)) for i, s in enumerate(lines)
                if normalize_heading(s) in expected]
    headings = [h for _, h in detected]
    missing = [h for h in expected if h not in headings]
    duplicates = sorted({h for h in headings if headings.count(h) > 1})
    ordered = headings == expected
    empty = []
    for j, (line_index, heading) in enumerate(detected):
        next_index = detected[j + 1][0] if j + 1 < len(detected) else len(lines)
        body = lines[line_index+1:next_index]
        if not any(x.strip() for x in body):
            empty.append(heading)
    return {
        'format_ok': not missing and not duplicates and ordered and not empty,
        'section_hits': len(expected) - len(missing),
        'sections_total': len(expected),
        'missing': missing,
        'out_of_order': not ordered and not missing and not duplicates,
        'duplicate_sections': duplicates,
        'empty_sections': empty,
    }


# Мини-тесты проверяющей функции (НЕ ответы модели).
examples = [
    ('корректный Markdown', '### Возможные причины\n- Перелив.\n### Что проверить\n- Влажность грунта.\n### Что делать\n- Проверить дренаж.', True),
    ('корректные жирные заголовки', '**Условия:**\n- Свет.\n**Рекомендации:**\n- Полив.', True),
    ('фразы внутри текста', 'В таких условиях рекомендации зависят от растения. Возможные причины бывают разные; нужно понять, что проверить и что делать.', False),
    ('заголовок внутри предложения', 'Возможные причины: перелив. Что проверить: грунт. Что делать: сократить полив.', False),
    ('пустой раздел', 'Возможные причины\nЧто проверить\n- Грунт.\nЧто делать\n- Осмотреть корни.', False),
    ('перепутан порядок', 'Что проверить\n- Грунт.\nВозможные причины\n- Перелив.\nЧто делать\n- Осмотреть корни.', False),
]
for title, sample, expected_ok in examples:
    task = 'care' if title == 'корректные жирные заголовки' else 'diagnosis'
    result = format_check(sample, task)
    assert result['format_ok'] == expected_ok, f'{title}: {result}'
    print(title, '→', result['format_ok'])

for item in quick_run:
    print(item['id'], format_check(item['answer'], item['task']))


## 6. Полный тест

Включается флагом `RUN_FULL_EVAL`.


In [ ]:
RUN_FULL_EVAL = False  # установите True после санити-чека

if RUN_FULL_EVAL:
    expected_by_id = {item['id']: item for item in EVAL_QUESTIONS}
    completed = {}
    if baseline_path.exists():
        for line_no, line in enumerate(baseline_path.read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip():
                continue
            try:
                prev = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError(f'Повреждён JSONL на строке {line_no}: {exc}') from exc
            key = prev.get('id')
            if key not in expected_by_id:
                raise RuntimeError(f'Неожиданный вопрос в сохранённом прогоне: {key}')
            source = expected_by_id[key]
            if (prev.get('run_id') != RUN_ID or
                prev.get('question') != source['question'] or
                prev.get('task') != source['task']):
                raise RuntimeError(f'Несовпадение сохранённого ответа для {key}; данные не смешиваем')
            if key in completed:
                raise RuntimeError(f'Дублированный ID в JSONL: {key}')
            completed[key] = prev
    print(f'Уже готовы: {len(completed)}/{len(EVAL_QUESTIONS)}')
    for idx, item in enumerate(EVAL_QUESTIONS, 1):
        if item['id'] in completed:
            continue
        start = time.perf_counter()
        response = answer_question(item['question'])
        checked = format_check(response, item['task'])
        result = {**item, 'answer': response,
                  'seconds': round(time.perf_counter()-start, 2),
                  **checked, 'run_id': RUN_ID,
                  'model_id': MODEL_ID, 'model_revision': model_commit}
        with baseline_path.open('a', encoding='utf-8') as f:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
        print(f'[{idx}/{len(EVAL_QUESTIONS)}] {item["id"]}: format_ok={checked["format_ok"]}')
    print('Файл с ответами:', baseline_path)
else:
    print('Полный прогон не запущен. Установите RUN_FULL_EVAL = True.')


## 7. Метрики и график


In [ ]:
if baseline_path.exists():
    records = [json.loads(s) for s in baseline_path.read_text(encoding='utf-8').splitlines()
               if s.strip()]
    expected_by_id = {item['id']: item for item in EVAL_QUESTIONS}
    if (len(records) != len(EVAL_QUESTIONS) or
        len({r['id'] for r in records}) != len(EVAL_QUESTIONS) or
        any(r.get('run_id') != RUN_ID or
            r['id'] not in expected_by_id or
            r['question'] != expected_by_id[r['id']]['question'] or
            r['task'] != expected_by_id[r['id']]['task'] for r in records)):
        print(f'Нет полного согласованного прогона: {len(records)}/{len(EVAL_QUESTIONS)}. '
              'Сначала закончите шаг 5; итоговые метрики не сформированы.')
    else:
        # Пересчитываем оценки строго по текущей версии функции.
        for r in records:
            r.update(format_check(r['answer'], r['task']))
        df = pd.DataFrame(records).sort_values('id')
        df.to_csv(RUN_DIR / 'baseline_answers.csv', index=False, encoding='utf-8-sig')
        summary = df.groupby('task', as_index=False).agg(
            count=('format_ok', 'size'), pass_rate=('format_ok', 'mean'),
            avg_section_hits=('section_hits', 'mean'), avg_seconds=('seconds', 'mean'))
        total = {'task': 'ALL', 'count': len(df),
                 'pass_rate': float(df['format_ok'].mean()),
                 'avg_section_hits': float(df['section_hits'].mean()),
                 'avg_seconds': float(df['seconds'].mean())}
        summary = pd.concat([summary, pd.DataFrame([total])], ignore_index=True)
        summary.to_csv(RUN_DIR / 'baseline_metrics.csv', index=False,
                       encoding='utf-8-sig')
        display(summary)
        ax = summary.plot.bar(x='task', y='pass_rate', legend=False,
                              ylim=(0, 1), rot=0,
                              title='Baseline: соблюдение структуры по типам запросов')
        ax.set_ylabel('Доля ответов, прошедших проверку')
        plt.tight_layout()
        plt.savefig(RUN_DIR / 'baseline_format_rate.png', dpi=160)
        plt.show()
        print('Готово. Файлы результатов:', RUN_DIR)
else:
    print('Сначала выполните полный прогон в шаге 5.')


## 8. Шаблон оценки качества


In [ ]:
JUDGE_TEMPLATE = """Ты независимый оценщик ответов о комнатных растениях.
Оцени только представленный ответ, не зная, какая модель его создала.
По шкале 1–5 оцени:
(1) fact_score — насколько корректны сведения;
(2) action_score — насколько полезны и безопасны следующие действия.
Не штрафуй за отсутствие категоричного диагноза, когда данных недостаточно.
Укажи конкретные ошибки и недостающие уточнения.
Возвращай только JSON с ключами fact_score, action_score, errors, rationale.

Вопрос: {question}
Ответ: {answer}"""
(RUN_DIR / 'judge_prompt_template.txt').write_text(JUDGE_TEMPLATE, encoding='utf-8')
print('Шаблон LLM-судьи сохранён в:', RUN_DIR)
print(JUDGE_TEMPLATE)


## 9. Экспорт результатов


In [ ]:
import shutil
zip_path = shutil.make_archive('/content/plant_advisor_baseline_results','zip', root_dir=PROJECT)
print(zip_path)
from google.colab import files
files.download(zip_path)